# MEDAL: fixed-budget search, complete metrics, then optional batching

[Open in Colab](https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/MEDAL_MCTS_Demo.ipynb)

This is the MEDAL counterpart of [AB_MEDAL_MCTS_Colab.ipynb](AB_MEDAL_MCTS_Colab.ipynb). Both use **the first frozen development question, depth 7, search budget 256, batch size 1**, the same two-shot prompt/tokenization, immediate IG, deepest-state selection, Confidence T=0 completion, and offline metrics. Only initialization differs: MEDAL uses UCB selection and retained-sibling expansion rounds; AB uses Student-t GEN/CONT.

Validate the serial search and offline logs before enabling the five-question development run. The final optional section checks padding and full-run parity at batch sizes 2 and 4 before measuring throughput. `SEARCH_BUDGET` counts sequence evaluations; `BATCH_SIZE` counts independent questions. Batching speedup is a separate implementation result.

See [the notebook alignment guide](../../docs/medal_notebook.md) and [shared experimental protocol](../../docs/experiments.md).

## Setup — once per runtime

Use the pinned LLaDA dependency profile and a CUDA runtime with bfloat16 support. The existing checkout and Drive cache flow is preserved. Validate the frozen manifest before loading weights. All later cells reuse one model.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")

In [ ]:
import json
import torch
from IPython.display import Image, Markdown, display
from dlm_project.models.profiles import get_profile
from dlm_project.experiments.paths import create_run_directory
from dlm_project.decoding.medal_demo import show_candidates
from dlm_project.decoding import search_audit
from dlm_project.tasks.gsm8k import ab_development as dev
from dlm_project.tasks.gsm8k.ab_batch_study import run_study

METHOD = "medal"  # Only initialization differs between the paired notebooks.
CONDITION = "A"  # A=(7,256), B=(7,512), C=(10,512), D=(10,1024)
SAMPLE_INDEX = 0
BATCH_SIZE = 1
ctx = dev.context(method=METHOD, stage="dev", condition=CONDITION, indices=(SAMPLE_INDEX,))
SEARCH_BUDGET = ctx["search"]["max_model_calls"]
print("Frozen question:", ctx["examples"][0]["example_id"])
print("Shared protocol:", ctx["common_protocol_hash"])
print("Search settings:", ctx["search"])

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a CUDA runtime supporting the shared bfloat16 model configuration.")
print("GPU:", torch.cuda.get_device_name(0))
model_name = get_profile("llada")["repo_id"].split("/")[-1]
stored_models = [Path("/content/models") / model_name, DRIVE_ROOT / "models" / model_name]
MODEL_PATH = next((str(path) for path in stored_models if (path / "config.json").is_file()),
                  get_profile("llada")["repo_id"])
SINGLE_OUTPUT, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / f"results/{METHOD}/development")})

In [ ]:
llada = dev.load_model(ctx, model_path=MODEL_PATH, device="cuda",
                       local_files_only=Path(MODEL_PATH).is_dir())
prepared = dev.prepare(llada, ctx)
item = prepared[0]
ctx["setup"]["warmup"] = dev.warmup(llada, prepared, ctx)
print("Setup measurements (excluded from decoding):", ctx["setup"])

## 1. Prompting and initial state

The shared pens/muffins two-shot template is passed through the pinned tokenizer's existing user chat format. No demonstrations are fetched and no model forward generates a plan. The current reference answer is used only for scoring. Search starts with these exact prompt IDs followed by 256 mask tokens.

In [ ]:
print(item["text"])
print("Template hash:", item["prompt_template_hash"])
print("Rendered-prompt hash:", item["prompt_hash"])
print("Token-ID hash:", item["prompt_ids_hash"])
print("Prompt tokens:", len(item["prompt_ids"]), "| output masks: 256 | seed:", item["seed"])

## 2. Search: selection → expansion → simulation → backup

Initialization calls the existing MEDAL `MCTSSearch`. UCB uses empirical mean reward plus exploration constant 0.01. After choosing a parent, expand its retained actions in order (up to K2=5 children) before reselecting. Each edge commits one output token. Action filtering uses K1=3, K2=5, margin parameter 10, and the same mask/EOS/EOT exclusions as AB-MEDAL.

Immediate reward is `(H(parent)-H(child))/H(parent)` when parent entropy exceeds `1e-8`, otherwise zero. Negative rewards are retained. Each successful child reward is backed up once through that child and its ancestors. MEDAL has no GEN/CONT decisions or Bayesian posteriors.

Search performs no full rollouts. Root analysis and each child evaluation consume one sequence NFE; finding three candidates does not stop search. Stop only at the budget or exhausted frontier, including a budget reached partway through a sibling round. Errors and interruptions retain partial artifacts and propagate.

Optional cross-question batching pauses at evaluation boundaries while retaining the selected parent until its sibling round ends. It therefore preserves ordinary MEDAL selection order; it does not switch to one-child reselection.

In [ ]:
initialization = dev.initialize(llada, item, ctx, SINGLE_OUTPUT)
completion = response = record = None
print("Stop:", initialization.search.stop_reason)
print("Search NFEs:", initialization.result.stats["search_model_calls"])
print("Selected depth:", initialization.result.selected.depth)

### Deepest candidates and final shortlist

Collect every distinct state at the deepest successfully evaluated depth. The first occurrence supplies its final statistics. Rank by empirical `Q = reward_sum / visits`, with ascending node ID for ties; retain at most three and complete rank 1. Posterior locations and sampled GEN/CONT scores do not rank candidates.

Fallback means selected depth below requested depth. One/two-item shortlists and the analyzed root are valid. The root has zero visits and Q=0. Requested-depth candidate count is reported separately.

In [ ]:
show_candidates(initialization, tokenizer=llada.tokenizer)
print("All deepest candidates:", initialization.result.stats["candidate_count"])
print("Requested-depth candidates:", initialization.result.stats["target_depth_candidate_count"])
print("Shortlist:", initialization.result.stats["candidate_shortlist"])
print("Fallback:", initialization.result.stats["fallback_decode_used"])
ranked_candidates = sorted(initialization.result.stats["candidate_evaluations"],
                           key=lambda c: (-c["q_value"], c["node_id"]))
candidate_table = ["| Rank | Node | Depth | Visits | Reward sum | Immediate IG | Empirical Q |",
                   "| --- | --- | --- | --- | --- | --- | --- |"]
for rank, candidate in enumerate(ranked_candidates, 1):
    candidate_table.append("| " + " | ".join(str(v) for v in (
        rank, candidate["node_id"], candidate["depth"], candidate["visits"],
        candidate["reward_sum"], candidate["immediate_gain"], candidate["q_value"])) + " |")
display(Markdown("\n".join(candidate_table)))


## 3. LLaDA Confidence completion and scoring

Use temperature 0 and preserve every prompt/search commitment. Each forward fills one remaining output mask, so completion NFEs equal `256 - selected_depth`. EOS/EOT are allowed during completion and do not stop fixed-length generation. Score only the completed output prefix before the first terminator, using strict `####` parsing.

In [ ]:
completion = dev.complete(llada, item, initialization, ctx, SINGLE_OUTPUT)
record = dev.finish_single(llada, item, ctx, initialization, completion, SINGLE_OUTPUT)
response = record["decoded"]["scoring_text"]
print(response)
print("Score:", record["score"])
print("Artifacts:", SINGLE_OUTPUT)

## 4. All metrics for n=1, reconstructed offline

This includes accuracy/parsing; maximum/selected/mean depth; branching min/mean/max and expanded parents; deepest/target candidates and unique states; negative IG count/percentage, IG distribution and selected Q; NFEs per depth and root subtree; GEN/CONT and forced decisions; duplicates, target success, fallback, and costs.

Branching counts evaluated children per expanded parent. Root analysis is depth 0 and is separate from subtree allocation. Negative IG and duplicate percentages use evaluated children; GEN/CONT percentages use decisions, because one child evaluation may involve multiple CONT decisions. Timing excludes setup, warm-up, tokenization, scoring, offline analysis, plots, and artifact writes.

MEDAL reports GEN/CONT and forced-decision metrics as unavailable because it uses UCB. The common tree, reward, allocation, candidate, and decoding metrics use identical definitions for both methods.

In [ ]:
offline_metrics = search_audit.load(SINGLE_OUTPUT)
for key, value in offline_metrics.items():
    assert record[key] == value, f"Offline mismatch: {key}"
assert sum(offline_metrics["nfes_per_depth"].values()) == record["search_nfes"]
assert offline_metrics["root_analysis_nfes"] + sum(offline_metrics["nfes_per_root_subtree"].values()) == record["search_nfes"]
assert record["total_nfes"] == record["search_nfes"] + record["completion_nfes"]
assert record["completion_nfes"] == 256 - record["selected_depth"]
dev.show_metrics(record)
dev.plot_metrics(SINGLE_OUTPUT)
display(Image(filename=str(SINGLE_OUTPUT / "compute_allocation.png")))
display(Image(filename=str(SINGLE_OUTPUT / f"{METHOD}_tree.png")))
for key, label in (("nfes_per_depth", "Depth"), ("nfes_per_root_subtree", "Root-child node")):
    table = [f"| {label} | Search sequence NFEs |", "| --- | --- |"]
    table.extend(f"| {identifier} | {nfes} |" for identifier, nfes in offline_metrics[key].items())
    display(Markdown("\n".join(table)))
print("Root-analysis NFEs (outside subtrees):", offline_metrics["root_analysis_nfes"])


## 5. Optional serial validation on all five development questions

Enable after inspecting the first run. This keeps the selected depth/budget, uses all five frozen development questions in manifest order, and reuses the loaded model. Every question gets independent search statistics and its fixed seed. Results are flushed after each question; interruptions and errors preserve partial summaries.

In [ ]:
RUN_SERIAL_DEV = False
serial_development = None
if RUN_SERIAL_DEV:
    dev_ctx = dev.context(method=METHOD, stage="dev", condition=CONDITION, indices=tuple(range(5)))
    dev_inputs = dev.prepare(llada, dev_ctx)
    dev_output, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / f"results/{METHOD}/dev5")})
    serial_development = dev.run_samples(llada, dev_inputs, dev_ctx, dev_output, batch_size=1)
    print(json.dumps(serial_development["summary"], indent=2))

## 6. Optional batch validation, then throughput at sizes 1 / 2 / 4

Enable only after successful five-question serial validation. Each batch contains independent questions, not siblings from one tree. The same first four development questions and fixed seeds are reused for every size. Search NFEs remain per question; actual model invocations and batch occupancy are reported separately.

Before timing, compare unpadded serial, padded serial, and batched predictions on saved root/selected states; then compare complete searches and answers. Discrete decisions, selected states, and output tokens must agree. IG/Q use `rtol=1e-4, atol=1e-5`. Attention masks alone are not a parity guarantee. Failed configurations are excluded from throughput results.

Three repetitions rotate batch-size order after separate warm-up. Reports include runtime spread, throughput/speedup, memory, occupancy, NFEs, correctness and fallback. OOMs are recorded without reducing batch size. Group wall time is used for speedup; per-question latencies overlap and must not be summed. Future MEDAL/AB algorithm comparisons must use matched execution conditions; serial remains the default.

In [ ]:
RUN_BATCH_STUDY = False
REPETITIONS = 3
batch_study = None
if RUN_BATCH_STUDY:
    if serial_development is None or serial_development["summary"]["status"] != "completed":
        raise RuntimeError("Complete the optional five-question serial validation section first.")
    batch_ctx = dev.context(method=METHOD, stage="dev", condition=CONDITION, indices=tuple(range(4)))
    batch_inputs = dev.prepare(llada, batch_ctx)
    batch_output, _ = create_run_directory({"output_dir": str(DRIVE_ROOT / f"results/{METHOD}/batching")})
    batch_study = run_study(llada, batch_inputs, batch_ctx, batch_output, repetitions=REPETITIONS)
    print(json.dumps(batch_study["summary"], indent=2))
    print("Validation:", json.dumps(batch_study["validation"], indent=2))
    if (batch_output / "batch_comparison.png").is_file():
        display(Image(filename=str(batch_output / "batch_comparison.png")))
    print("Batch artifacts:", batch_output)